# 第八讲 —— Continuation Value is All You Need(延续价值就是你需要的一切)

**异质性主体宏观经济学的计算方法**  
孙杰,多伦多大学

---

在第七讲中,我们用近似运动方程(Approximate Law of Motion, ALM)求解了 Krusell–Smith 模型:对 $K = \int b\, d\mu$ 使用一条单行的对数线性规则。这里我们同时放弃 ALM *与* 标量 $K$ 摘要。我们训练的神经网络(neural network)$\widehat{V}^{\mathrm{end}}_\theta$ 通过一个学习得到的广义矩(generalized moment)直接以完整的人口分布 $\Lambda$ 为条件。训练过程最小化外层 Bellman 残差(outer Bellman residual)。

## 0 · 环境设置

激活项目环境并加载相关包。**第八讲需要 `Flux`** 来构建神经网络;如果尚未安装,我们会自动添加。

In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()

using Flux
using Flux: Chain, Dense, swish, Adam, withgradient, mse
using HouseholdStages
using Printf
using Statistics
using LinearAlgebra: I
using Random: MersenneTwister, seed!
using Plots

## 1 · 沿用自第七讲的 K-S 模型设定

我们逐字复用第七讲的 `KSParams`、价格函数与有效劳动辅助函数。家庭模块依然是 *三阶段* 模拟链 —— `z_shock ∘ income ∘ savings` —— 也就是期内求解算子 $\Phi$。第七讲的四维链在末尾追加了 `K_evolve ∘ Z_shock`,用于编码主体所持的 ALM 信念;今天我们 *去掉* 这个尾巴。神经网络将直接编码主体对下一期 $(\Lambda, Z)$ 的预期。

In [ ]:
@kwdef struct KSParams
    β::Float64 = 0.96
    γ::Float64 = 1.0
    α::Float64 = 0.36
    δ::Float64 = 0.025
    z_grid::Vector{Float64} = [0.07, 1.0]
    P_z::Matrix{Float64}    = [0.6   0.4;
                               0.05  0.95]
    Z_vals::Vector{Float64} = [0.99, 1.01]
    P_Z::Matrix{Float64}    = [0.875 0.125;
                               0.125 0.875]
    N_w::Int       = 300
    w_min::Float64 = 0.0
    w_max::Float64 = 80.0
end

const p = KSParams()
N_z = length(p.z_grid)
N_Z = length(p.Z_vals)
@printf "β = %.3f, γ = %.2f, α = %.2f, δ = %.3f, N_w = %d, N_z = %d, N_Z = %d\n" p.β p.γ p.α p.δ p.N_w N_z N_Z

In [ ]:
"计算个体 z 过程的平稳均值"
function ks_effective_labor(P_z::AbstractMatrix, z_grid::AbstractVector)
    n = size(P_z, 1)
    A = P_z' - I(n)
    A[end, :] .= 1.0
    rhs = zeros(n); rhs[end] = 1.0
    π = A \ rhs
    return sum(z_grid .* π)
end

function ks_prices(K::Real, Z::Real, p::KSParams)
    L = ks_effective_labor(p.P_z, p.z_grid)
    r = p.α * Z * (K/L)^(p.α - 1) - p.δ
    w = (1 - p.α) * Z * (K/L)^p.α
    return (; r, w)
end

const L_eff = ks_effective_labor(p.P_z, p.z_grid)
@printf "L_eff = %.4f\n" L_eff

## 2 · 代码中的 $\Phi$:三阶段家庭链

第七讲的 `ks_household_2d` *就是* 我们的 $\Phi$。给定一个 $(b, z)$ 值函数 `V_end`,一次后向扫描得到 `V_start`;一次前向扫描把 `Λ_start` 推到 `Λ_end`。价格 $(r, w)$ 从 `env` 中读取。由于该模型在每一期是单主体的,$\Phi$ 对 $\Lambda$ 的依赖只通过价格体现 —— 而价格对 $\Lambda$ 的依赖只通过标量 $K = \int b\, d\mu$ 体现。

In [ ]:
_u_crra(c, ::Union{Val{1}, Val{1.0}}) = log(c)
_u_crra(c, ::Val{σ}) where σ = (c^(1 - σ)) / (1 - σ)
u_crra(c, valσ::Val) = c < 0 ? -Inf : _u_crra(c, valσ)

"构建家庭模块"
function ks_household_2d(p::KSParams)
    # 定义状态布局
    layout = StateLayout(
        StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length=p.N_w, spacing=:log)),
        StateAxis(:z, p.z_grid),
    )

    # 定义三个阶段
    z_shock = MarkovStage(layout; axis=:z, transition=p.P_z)
    income  = WealthChangeStage(layout;
        wealth_post=(cell; env) -> (1 + env.r) * cell.wealth + env.w * cell.z)
    savings = ConsumptionSavingsStage(layout; β=p.β,
        utility=(cell, c; env) -> u_crra(c, Val(p.γ)),
        monotone_search=:divide_conquer)
    
    # 构建家庭链
    hh = z_shock ∘ income ∘ savings

    # 定义矩
    return define_moments!(hh;
                           K_supplied=at_end(integrand=:wealth, reduce=sum))
end

hh = ks_household_2d(p)
@printf "Built 3-stage chain hh = z_shock ∘ income ∘ savings.\n"

## 3 · 在 $Z = 1$ 处的确定性 Aiyagari 稳态

我们需要从确定性问题中获得两个产物:

1. **$V_{\mathrm{ss}}(b, z)$** —— 作为预训练时跨各采样 $\Lambda$ 的目标。
2. **$\Lambda_{\mathrm{ss}}(b, z)$** —— 确定性稳态分布,既用作遍历采样器的初始条件,也用作预训练扰动的锚点。

这就是第七讲的 `aiyagari_steady_state_at_Z`,原样照搬。

In [ ]:
"求解 Aiyagari 稳态。"
function aiyagari_steady_state_at_Z(p::KSParams; Z::Float64=1.0, verbosity=0, maxiter=1000, rtol=1e-2, update_speed=0.01)

    # 构建家庭
    hh = ks_household_2d(p)

    # 初始猜测
    K = 12.0
    V, Λ = nothing, nothing

    # 迭代直至收敛
    K_err = Inf
    iters = 0

    while K_err > rtol
        # 把矩作为 env 的函数计算
        env = make_env(hh; ks_prices(K, Z, p)...)
        (;V, Λ, moments) = solve_steady_state_given_env!(hh, env; V_init=V, Λ_init=Λ, lambda_tol=1e-5, lambda_maxiter=50_000)
        (;K_supplied) = moments

        # 计算误差
        K_err = abs(K_supplied - K) / K
        verbosity >= 1 && @printf "  iter %d: K = %.3f → K_sup = %.3f\n" iters K K_supplied

        # 更新 K
        K += update_speed * (K_supplied - K)

        # 检查最大迭代次数
        iters += 1
        iters >= maxiter && error("aiyagari_steady_state_at_Z: did not converge")
    end

    return (; K, Λ, V, iters)
end

println("Solving deterministic Aiyagari SS at Z = 1.0...")
@time det_ss = aiyagari_steady_state_at_Z(p; verbosity=1)
const K_bar = det_ss.K
const V_ss  = det_ss.V
const Λ_ss  = det_ss.Λ
@printf "K̄ = %.4f in %d iters; sum(Λ_ss) = %.6f\n" K_bar det_ss.iters sum(Λ_ss)

## 4 · 重新表述:从 $K$-ALM 到 $V_\theta(b, z, \Lambda, Z)$

第七讲的 K-S 基线通过在一个小 $K$ 网格与两点 $Z$ 网格上列表 $V[b, z, K, Z]$ 求解一个四维 Bellman 方程,并假设主体借助一条对数线性 ALM 预测 $K_{t+1}$。标量 $K$ 的 *全部用途* 就是记账:K-S 需要一个可处理的对象来预测,而参数化拟设也只能承载一个数。

我们不必接受这种妥协。神经网络可以直接以完整分布 $\Lambda$ 为条件:

$$V_\theta : (b, z, \Lambda, Z) \;\longmapsto\; v \in \mathbb{R}.$$

我们想要的函数就是满足 Bellman 方程的那一个。等价地说:它是 §7 中定义的 **外层 Bellman 算子** $\mathcal{L}$ 的不动点。于是我们最小化均方 **外层 Bellman 残差**:

$$\mathcal{J}(\theta) \;=\; \mathbb{E}_{\Lambda^{\mathrm{start}}}\!\bigl[\bigl\|V_\theta - \mathcal{L} V_\theta\bigr\|^2\bigr].$$

挑战在于 $\Lambda$ 是高维的:一个 `N_w × N_z = 80 × 2 = 160` 单元的场。我们没法把它直接喂进一个全连接网络还指望学到任何有意义的东西。§5 解释了 **广义矩**(generalized moment)构造如何处理这一问题 —— 其架构在值函数头看到 $\Lambda$ 之前,先把它压缩成一个小型 *学习得到* 的向量。

## 5 · 神经网络 $V_\theta$:广义矩架构

$V_\theta(b, z, \Lambda, Z)$ 分解为四部分,呼应 Han–Yang–E (2025):

1. **`pop_agg_net`** $: (b, z) \mapsto \mathbb{R}^{k}$ —— 把每个个体单元嵌入到一个 $k$ 维向量。同一个网络作用于每个单元。
2. **广义矩** $\mathrm{GM}(\Lambda) \;=\; \sum_{(b, z)} \Lambda(b, z) \cdot \texttt{pop\_agg\_net}(b, z) \;\in\; \mathbb{R}^{k}$ —— 单元嵌入按人口加权的平均。一次对 $\mathrm{vec}(\Lambda)$ 的矩阵乘法。**这就是完整分布被压缩的方式:压缩为一个小的 $k$ 维学习向量,而不是手挑的标量。**
3. **`V_pre`** $: (b, z) \mapsto \mathbb{R}^{m}$ —— 单元级的特征,将与 $\mathrm{GM}$ 加性地合并。同一个网络作用于每个单元。
4. **`V_post`** $: \mathbb{R}^{m + N_Z} \mapsto \mathbb{R}$ —— 把 `V_pre` 的单元特征、投影后的 $\mathrm{GM}$ 与 $Z$ 的 one-hot 编码合在一起,读出 $\widehat{V}^{\mathrm{end}}$。

合并步骤是 **加性的**:`V_pre(cell) + gm_post(GM(Λ))`。然后拼接 $Z$ 的 one-hot,送入 `V_post`。加性合并迫使网络以与 $(b, z)$ 依赖相同的尺度来表达 $\Lambda$ 依赖,从而稳定训练(Han 等 2025 §4)。

下面把瓶颈维度 $k$ 设为 16,这正是该方法能够扩展的关键:我们用一个 $k$ 维输入代替了 `N_w × N_z` 维输入,并 *学习* 这个摘要,而不是手挑 $K$。

In [ ]:
# 一次性预构造 (2, N_w * N_z) 形状的个体特征矩阵。每一列
# 是一个 (wealth, z) 单元的归一化 (b, z) 坐标。
function build_hh_feat(p::KSParams)
    b_norm = collect(range(0f0, 1f0, length=p.N_w))
    z_norm = collect(range(0f0, 1f0, length=length(p.z_grid)))
    feat = zeros(Float32, 2, p.N_w * length(p.z_grid))
    col = 1
    for j in 1:length(p.z_grid), i in 1:p.N_w
        feat[1, col] = b_norm[i]
        feat[2, col] = z_norm[j]
        col += 1
    end
    return feat
end

const HH_FEAT = build_hh_feat(p)
const N_CELLS = size(HH_FEAT, 2)

# 预构造每个 Z_idx 对应的 one-hot Z 行。形状 (N_Z, N_CELLS),便于
# 在 V_post 中与 (mid, N_CELLS) 的隐藏特征直接 vcat 在一起。
const Z_ROWS = [repeat(Float32.(1:N_Z .== Z_idx), 1, N_CELLS) for Z_idx in 1:N_Z]

In [ ]:
# V_θ : (Λ, Z_idx) ↦ 形状为 N_w × N_z 的 V_end 矩阵。V_post 的输出
# 偏置初始化为大致的 V_ss 均值,以免内层后向扫描从随机初始化处
# 发散;§10 的预训练会从此基础上进一步精炼。
struct VNet{A, P, G, V}
    pop_agg_net::A    # 2 → gm_dim          每个单元的 GM 嵌入
    V_pre::P          # 2 → mid             V 头使用的每单元特征
    gm_post::G        # gm_dim → mid        把 GM 投影到 mid 空间
    V_post::V         # mid + N_Z → 1       合并并结合 Z,读出 V_end
end

function VNet(; gm_dim::Int=16, mid::Int=32, seed::Int=9483, v_bias::Float32=20f0)
    seed!(seed)
    pop_agg_net = Chain(
        Dense(2 => mid, swish),
        Dense(mid => gm_dim),
    )
    V_pre = Chain(
        Dense(2 => mid, swish),
        Dense(mid => mid, swish),
    )
    gm_post = Dense(gm_dim => mid)
    V_post = Chain(
        Dense(mid + N_Z => mid, swish),
        Dense(mid => 1),
    )
    V_post.layers[end].bias .= v_bias
    return VNet(pop_agg_net, V_pre, gm_post, V_post)
end

Flux.@layer VNet

function (net::VNet)(Λ::AbstractMatrix, Z_idx::Int, p::KSParams)
    pop_emb = net.pop_agg_net(HH_FEAT)        # (gm_dim, N_CELLS)
    h_pre   = net.V_pre(HH_FEAT)              # (mid, N_CELLS)
    gm      = pop_emb * Float32.(vec(Λ))      # (gm_dim,)   —— 按人口加权的均值
    h_gm    = net.gm_post(gm)                 # (mid,)
    h       = h_pre .+ h_gm                   # 广播 → (mid, N_CELLS)
    pred    = net.V_post(vcat(h, Z_ROWS[Z_idx]))   # (1, N_CELLS)
    return reshape(pred, p.N_w, length(p.z_grid))
end

vnet = VNet()
V0 = vnet(Λ_ss, 1, p)
@printf "VNet output at (Λ_ss, Z=1): shape = %s, range = [%.3f, %.3f]\n" string(size(V0)) minimum(V0) maximum(V0)

## 6 · 在 $V_\theta$ 下的一次内层求解

给定一个总量状态 $(\Lambda, Z)$:

1. 计算 $V_\theta(\Lambda, Z)$,得到一个 `N_w × N_z` 的 $V^{\mathrm{end}}$ 数组。
2. 以价格 $(r, w) = $ `ks_prices(K(Λ), Z, p)` 构造 `env`,其中 $K(\Lambda) = \int b\, d\Lambda$。价格对 $\Lambda$ 的依赖只通过 $K$ —— 这是生产技术决定的,与网络架构无关。
3. 沿三阶段链做一次后向扫描,返回 $V^{\mathrm{start}}$。

这就是讲稿(§3)中的 $\Phi_V$。

In [ ]:
# 在二维分布上对财富(K 矩)进行积分。
function integrate_K(hh, Λ::AbstractMatrix)
    b_grid = axisvalues(first(hh.spec.stages).input_layout.axes[1])
    return sum(Λ[i, j] * b_grid[i] for i in axes(Λ, 1), j in axes(Λ, 2))
end

# 在给定 (Λ, Z) 下,沿 V_θ 跑一次三阶段链的后向扫描。
# 返回期初的 V(形状为 N_w × N_z 的矩阵)。
function inner_solve_backward(hh, vnet::VNet, Λ::AbstractMatrix, Z_idx::Int, p::KSParams)
    V_end = Float64.(vnet(Λ, Z_idx, p))
    env   = make_env(hh; ks_prices(integrate_K(hh, Λ), p.Z_vals[Z_idx], p)...)
    return backward!(hh, V_end, env)
end

V_start_demo = inner_solve_backward(hh, vnet, Λ_ss, 1, p)
@printf "Inner backward at (Λ_ss, Z=bad): V_start shape = %s, range = [%.3f, %.3f]\n" string(size(V_start_demo)) minimum(V_start_demo) maximum(V_start_demo)

## 7 · 前瞻算子 $\mathcal{L} V_\theta$

对于单个采样 $\Lambda^{\mathrm{start}}_i$、总量索引 $Z_i$:

1. 计算 $V_\theta(\Lambda^{\mathrm{start}}_i, Z_i)$,得到 $V^{\mathrm{end}}$ —— 即主体对期末延续价值的当前信念。
2. 通过 $\Phi$ 把 $\Lambda^{\mathrm{start}}_i$ 推进一期,得到 $\Lambda^{\mathrm{end}}_i$。
3. 对每个下期总量状态 $Z_{\mathrm{next}}$:
   - 在 $(\Lambda^{\mathrm{end}}_i, Z_{\mathrm{next}})$ 上跑一次 $\Phi_V$,得到 $V^{\mathrm{start}}_{\mathrm{next}}$。
4. 用 $P_Z[Z_i, \cdot]$ 对 $Z_{\mathrm{next}}$ 取期望。

注:在此设定下,$\Omega(\Lambda^{\mathrm{end}}, Z')$ 是恒等的 —— 总量冲击 $Z'$ 影响下期的价格,从而影响 *下期* 的 $\Phi$,但不影响期界处的 $\Lambda$。因此 $\Lambda^{\mathrm{start}}_{i+1} = \Lambda^{\mathrm{end}}_i$。

结果即 **前瞻标签** $\widehat{V}^{\mathrm{end}}_i \equiv (\mathcal{L} V_\theta)(\Lambda^{\mathrm{start}}_i, Z_i)$。

In [ ]:
# 计算前瞻标签 LV_θ(Λ_start, Z) —— 见讲稿 §7。
# 注:需要一个 *已预训练* 的 V_θ;在随机网络下,储蓄策略可能
# 退化,而 forward! 可能产生坍缩的 Λ_end。我们在 §10 的预训练
# 之后再演示这个函数。
function lookahead(hh, vnet::VNet, Λ_start::AbstractMatrix, Z_idx::Int, p::KSParams)
    # 用神经网络得到当前总量下的 V_end
    V_end_now = Float64.(vnet(Λ_start, Z_idx, p))
    
    # 向前模拟一期。
    env_now   = make_env(hh; ks_prices(integrate_K(hh, Λ_start), p.Z_vals[Z_idx], p)...)
    _         = backward!(hh, V_end_now, env_now)         # 安置储蓄策略
    Λ_end     = forward!(hh, Λ_start)

    # 对每个 Z_next 再次前向模拟并后向迭代,然后取期望。
    V_end_label = zeros(Float64, p.N_w, length(p.z_grid))
    for Z_next in 1:N_Z
        V_start_next = inner_solve_backward(hh, vnet, Λ_end, Z_next, p)
        V_end_label .+= p.P_Z[Z_idx, Z_next] .* V_start_next
    end
    return (; V_end_label, Λ_end)
end

## 8 · 外层 Bellman 残差损失

对一批样本 $\{(\Lambda^{\mathrm{start}}_i, Z_i)\}$,先在 `withgradient` *之外* 计算一次前瞻标签 $\widehat{V}^{\mathrm{end}}_i$,然后用均方损失把 $V_\theta$ 拟合到它上面。这就是标准的半梯度(semi-gradient)做法 —— 我们不对 $\Phi$ 求导。

In [ ]:
# 通过对每个样本运行前瞻算子来构造训练目标。
# 返回一个 (Λ, Z_idx, target) 元组的向量;目标是阻断梯度的。
function build_targets(hh, vnet::VNet, batch, p::KSParams)
    targets = NamedTuple{(:Λ, :Z_idx, :target), Tuple{Matrix{Float64}, Int, Matrix{Float32}}}[]
    for (Λ_start, Z_idx) in batch
        out = lookahead(hh, vnet, Λ_start, Z_idx, p)
        push!(targets, (; Λ=copy(Λ_start), Z_idx, target=Float32.(out.V_end_label)))
    end
    return targets
end

# 一批 (Λ, Z, target) 元组上的外层 Bellman 残差损失。
function bellman_residual_loss(vnet::VNet, targets, p::KSParams)
    L = 0f0
    for (Λ, Z_idx, target) in targets
        pred = vnet(Λ, Z_idx, p)
        # 用目标自身的尺度做归一化,使损失无量纲。
        scale = max(var(target), 1f-6)
        L += sum((pred .- target).^2) / (length(pred) * scale)
    end
    return L / length(targets)
end

## 9 · 采样 $\Lambda^{\mathrm{start}}$

类 MCMC 方式:在 *当前* $V_\theta$ 下模拟模型,丢弃预热段(burn-in),再做子采样。采样器跟随网络 —— 训练早期,样本是从尚未训练好的网络所诱导的轨迹中抽出的;随着 $\theta$ 改进,分布也随之改进。

In [ ]:
# 从总量 Markov 链中抽取一期的 Z' | Z。
function sample_Z(Z_idx::Int, P_Z::AbstractMatrix, rng)
    probs = P_Z[Z_idx, :]
    u = rand(rng)
    s = 0.0
    for j in eachindex(probs)
        s += probs[j]
        u <= s && return j
    end
    return length(probs)
end

# 在已实现的 (Λ_t, Z_t) 下,沿 V_θ 向前模拟一期。先从 V_θ 重新
# 安置储蓄策略,然后返回 Λ_{t+1}(即 Λ_end)。
function sim_one_step(hh, vnet::VNet, Λ::AbstractMatrix, Z_idx::Int, p::KSParams)
    V_end = Float64.(vnet(Λ, Z_idx, p))
    env   = make_env(hh; ks_prices(integrate_K(hh, Λ), p.Z_vals[Z_idx], p)...)
    backward!(hh, V_end, env)
    return forward!(hh, Λ)
end

# 沿 V_θ 诱导的遍历路径,类 MCMC 地采样 (Λ_start, Z)。
function sample_ergodic(hh, vnet::VNet, p::KSParams;
                        T::Int=200, burn::Int=100, every::Int=2,
                        rng=MersenneTwister(8421))
    Λ      = copy(Λ_ss)
    Z_idx  = 1
    batch  = Tuple{Matrix{Float64}, Int}[]
    for t in 1:T
        Λ     = sim_one_step(hh, vnet, Λ, Z_idx, p)
        Z_idx = sample_Z(Z_idx, p.P_Z, rng)
        if t > burn && (t - burn) % every == 0
            push!(batch, (copy(Λ), Z_idx))
        end
    end
    return batch
end

## 10 · 预训练:把确定性 $V_{\mathrm{ss}}$ 平铺到 $(\Lambda, Z)$ 上

随机初始化的 $V_\theta$ 经常导致期内求解在数值上失败(消费跑出网格、储蓄策略退化)。一个便宜的修补办法是:在 $\Lambda$ 的一个 *云集*(对 $\Lambda_{\mathrm{ss}}$ 做扰动)与所有 $Z$ 上,预训练 $V_\theta$ 去匹配确定性稳态值函数 $V_{\mathrm{ss}}$。网络会落到一个合理的流形上;之后的残差训练循环会从此继续精修。

要点 *不在于* $V_{\mathrm{ss}}$ 是任意 $\Lambda$ 下的正确答案 —— 除了在 $\Lambda = \Lambda_{\mathrm{ss}}, Z = 1$ 处恰好成立以外,它都不是。要点在于它是一个合理的 *常数* 基线,能让网络在真正训练之前先到对的尺度和形状。

In [ ]:
# 廉价的 Λ 扰动:乘性噪声 + 重新归一化。用于在预训练时
# 围绕 Λ_ss 给出一小片合理分布的云集。
function perturb_Λ(Λ::AbstractMatrix, σ::Float64, rng)
    Λp = Λ .* (1 .+ σ .* randn(rng, size(Λ)))
    Λp = max.(Λp, 0)
    return Λp ./ sum(Λp)
end

# 在 (Λ, Z) 的一个云集上,预训练 V_θ 拟合确定性稳态 V_ss,
# 该云集横跨所有 Z。返回损失历史。
function pretrain_to_ss!(vnet::VNet, V_ss_target::AbstractMatrix, p::KSParams;
                         epochs::Int=300, lr::Float64=5e-3,
                         n_Λ::Int=6, σ::Float64=0.10, seed::Int=4747)
    opt    = Flux.setup(Adam(lr), vnet)
    target = Float32.(V_ss_target)
    rng    = MersenneTwister(seed)
    hist   = Float64[]
    for epoch in 1:epochs
        Λ_cloud = [perturb_Λ(Λ_ss, σ, rng) for _ in 1:n_Λ]
        loss, grads = withgradient(vnet) do net
            L = 0f0
            for Λ in Λ_cloud, Z_idx in 1:N_Z
                pred = net(Λ, Z_idx, p)
                L += sum((pred .- target).^2)
            end
            L / (n_Λ * N_Z * length(target))
        end
        Flux.update!(opt, vnet, grads[1])
        push!(hist, Float64(loss))
    end
    return hist
end

println("Pretraining V_θ on V_ss across a Λ-cloud...")
@time pre_hist = pretrain_to_ss!(vnet, V_ss, p; epochs=300)
@printf "Pretrain loss: %.4e → %.4e (over %d epochs)\n" pre_hist[1] pre_hist[end] length(pre_hist)

## 11 · 训练循环

一个 epoch 就是:在当前 $V_\theta$ 下采样一批新样本,计算前瞻标签(stop-gradient),对均方残差走一步 Adam。这是最简洁的半梯度形式;`reference_materials/example_usage/stochastic_transition/` 中的优化实现使用了余弦退火学习率、更大的批量、每个样本多步梯度,以及逐位置的反向传播(backprop)。

In [ ]:
# 外层 Bellman 残差训练循环。返回损失历史。
function train!(vnet::VNet, hh, p::KSParams;
                epochs::Int=50, lr::Float64=1e-3,
                T::Int=200, burn::Int=100, every::Int=4)
    opt  = Flux.setup(Adam(lr), vnet)
    hist = Float64[]
    rng  = MersenneTwister(1729)
    for epoch in 1:epochs
        batch   = sample_ergodic(hh, vnet, p; T, burn, every, rng=MersenneTwister(rand(rng, UInt32)))
        targets = build_targets(hh, vnet, batch, p)
        loss, grads = withgradient(vnet) do net
            bellman_residual_loss(net, targets, p)
        end
        Flux.update!(opt, vnet, grads[1])
        push!(hist, Float64(loss))
        if epoch == 1 || epoch % 5 == 0
            @printf "  epoch %3d / %3d: loss = %.4e (batch = %d)\n" epoch epochs loss length(batch)
        end
    end
    return hist
end

println("Running 50-epoch CVIAYN training (CPU demonstration, NOT a converged solve)...")
@time train_hist = train!(vnet, hh, p; epochs=50)

## 12 · 解读结果

两张图:损失曲线,以及在几个采样 $(\Lambda, Z)$ 处的 $V_\theta$ 与确定性稳态 $V$ 的比较。

In [ ]:
# 损失曲线(半对数)。一次收敛的求解会把它再压低好几个数量级。
plot(train_hist;
     yaxis=:log, lw=2, color=:steelblue,
     xlabel="epoch", ylabel="outer Bellman residual",
     title="CVIAYN training curve (50 epochs, CPU demo)",
     label="loss", size=(720, 360), legend=:topright)

In [ ]:
# 在三个 (Λ, Z) 设定下比较 V_θ(b, z=high) 与确定性稳态。
# 我们对 Λ_ss 作扰动,以展示 V_θ 会随分布输入而调整。
rng_show = MersenneTwister(12345)
Λ_high   = perturb_Λ(Λ_ss, 0.20, rng_show)
Λ_low    = perturb_Λ(Λ_ss, 0.20, rng_show)
b_grid = axisvalues(first(hh.spec.stages).input_layout.axes[1])
plt = plot(; xlabel="wealth b", ylabel="V_θ(b, z=high)",
           title="V_θ vs. deterministic SS V (z = high)",
           size=(720, 360))
plot!(plt, b_grid, V_ss[:, end]; lw=2, color=:black, ls=:dash, label="V_ss (det.)")
for (Λ_show, Z_show, lbl) in [(Λ_ss, 1, "Λ_ss, Z=bad"), (Λ_ss, 2, "Λ_ss, Z=good"), (Λ_high, 1, "perturbed Λ, Z=bad")]
    V_show = vnet(Λ_show, Z_show, p)
    plot!(plt, b_grid, V_show[:, end]; lw=2, label=lbl)
end
plt

## 13 · 与第七讲 K-S 基线的比较

完整的深度比较 —— 跑通第七讲的收敛 K-S 基线、在共同的 $(b, z, \Lambda, Z)$ 代理网格上计算 $V$ 的均方误差(MSE)、并在两种方法下计算 Den Haan 误差 —— 属于 `paper_code/`(横向比赛脚本)。本笔记本根据本次课时范围 *推迟* 处理它。

我们在这里廉价能做的:汇报一下 $V_\theta$ 最终落到了哪里,好让损失曲线图有点参照。

In [ ]:
# 指针单元 —— 真正的比较驱动脚本位于 paper_code/。
# 我们汇报一下 V_θ 最终落到了哪里,好让损失曲线单元有点参照。
final_loss = train_hist[end]
@printf "Final outer Bellman residual after %d epochs: %.4e\n" length(train_hist) final_loss
@printf "(For reference: the paper's Peking 2025 §5 figure reports K-S baseline plateau\n"
@printf " near 10^-3.2 and CVIAYN converged near 10^-4.4 in relative-variance units —\n"
@printf " each with several thousand epochs on a GPU.)\n"